# Module 9 · 文字 2：Subword Tokenization（2026 文本前處理的起點）

> **本節定位｜2026 主流核心**
> 所有現代語言模型（BERT、GPT、Llama、Qwen…）在「看到文字」之前，
> 第一步都是 **tokenization**：把字串切成 subword、再轉成整數 `input_ids`。
> 這是大模型資料前處理**最關鍵的第一步**。

## 學習目標
- 理解為什麼現代模型不用「詞」而用 **subword**（解決 OOV、控制詞彙表大小、跨語言）。
- 認識三大演算法：**BPE**（GPT 系）、**WordPiece**（BERT 系）、**Unigram/SentencePiece**（T5、多語）。
- 用 HuggingFace `AutoTokenizer` 完成 encode / decode，並產出標準張量 **`input_ids (B, L)`、`attention_mask (B, L)`**。
- 掌握 **padding / truncation / 批次化** 如何把不等長文本變成規整張量。
- 用 `tokenizers` **從零訓練一個 BPE tokenizer**，理解詞彙表是怎麼長出來的。
- 建立 **token 預算** 的成本直覺（token ≠ 單字）。

## 0. 資料結構設計：tokenizer 的輸入與輸出

這是 2026 文本資料的「標準介面」，下游所有模型都吃這個格式：

| 物件 | 形狀 | 型別 | 說明 |
|:--|:--|:--|:--|
| 原始輸入 | `(B,)` | `list[str]` | 一批 B 個字串 |
| `input_ids` | `(B, L)` | `int64` | 每個 token 在詞彙表中的 id；L = 序列長度 |
| `attention_mask` | `(B, L)` | `int64` | 1=真實 token，0=padding（告訴模型哪些要看） |
| `token_type_ids` | `(B, L)` | `int64` | （部分模型）句子 A/B 區隔，用於句對任務 |

**LLM 訓練/對話資料**的高層格式（下一步會在 `04_llm_data_formats` 深入）：
```jsonl
{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```
這些 `messages` 最終一樣會被 tokenizer（套用 chat template）攤平成 `input_ids (B, L)`。

In [ ]:
# 需要 multimodal extra：uv sync --extra multimodal
# AutoTokenizer.from_pretrained 只下載「tokenizer 設定/詞彙檔」（數 MB），不含模型權重，CPU 友善。
try:
    from transformers import AutoTokenizer
    HAS_HF = True
except Exception as e:
    HAS_HF = False
    print("未安裝 transformers，請先 `uv sync --extra multimodal`。本節示範說明仍可閱讀。")

## 1. 為什麼用 subword，而不是「詞」或「字元」？

| 切分粒度 | 詞彙表大小 | OOV（未見過的詞） | 序列長度 | 問題 |
|:--|:--|:--|:--|:--|
| 以「詞」 | 巨大（數十萬） | 嚴重：拼錯/新詞/專有名詞都變 `<UNK>` | 短 | 詞彙表爆炸 + 丟資訊 |
| 以「字元」 | 很小（數十~數百） | 幾乎沒有 | 很長 | 序列太長、難學語意 |
| **以 subword** | **適中（3 萬~25 萬）** | **幾乎沒有**（生字拆成已知碎片） | 適中 | **最佳折衷 → 現代標準** |

例：`tokenization` 可能被切成 `token` + `ization`；`ChatGPT` 切成 `Chat` + `G` + `PT`。
生字一律能用「已知碎片」拼出來，所以**沒有 OOV**。

## 2. 三大 subword 演算法（認識即可）

| 演算法 | 代表模型 | 核心思想 |
|:--|:--|:--|
| **BPE** (Byte-Pair Encoding) | GPT-2/3/4、Llama、Qwen | 從字元開始，反覆合併「最常一起出現」的相鄰對 |
| **WordPiece** | BERT、DistilBERT | 類似 BPE，但以「能最大化語料概率」挑選要合併的對 |
| **Unigram / SentencePiece** | T5、ALBERT、多語模型 | 先給大詞彙表，再逐步刪去貢獻最小的 token；語言無關（連空白都當字元） |

實務上你**很少自己選演算法**——載入哪個模型，就用它配套的 tokenizer（`AutoTokenizer` 會自動對應）。

## 3. 用 AutoTokenizer 實作：從字串到張量

In [ ]:
if HAS_HF:
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    text = "Tokenization is the first step of LLM preprocessing."

    enc = tok(text)
    print("subword tokens：")
    print(" ", tok.convert_ids_to_tokens(enc["input_ids"]))
    print("\ninput_ids：", enc["input_ids"])
    print("attention_mask：", enc["attention_mask"])
    print("\n[CLS]=101、[SEP]=102 是 BERT 自動加上的特殊 token")

**解讀**：
- 句子被切成 subword（注意 `preprocessing` 可能被拆成 `pre ##process ##ing`，`##` 表示「接續前一個 token」）。
- tokenizer 自動補上特殊 token（BERT 的 `[CLS]`/`[SEP]`）。
- 輸出就是模型要的整數序列 `input_ids` 與 `attention_mask`。

## 4. 批次 padding / truncation → 規整張量 `(B, L)`

真實訓練是「一批多句」，但每句長度不同。tokenizer 用 **padding** 補齊、**truncation** 截斷，
產出規整的 `(B, L)` 張量；`attention_mask` 記錄哪些是 padding。

In [ ]:
if HAS_HF:
    batch = [
        "Short text.",
        "A considerably longer sentence that demonstrates padding and truncation behaviour.",
        "Mid length example here.",
    ]
    out = tok(
        batch,
        padding=True,         # 補齊到批次中最長者
        truncation=True,
        max_length=16,        # 超過 16 個 token 就截斷
        return_tensors="pt",  # 直接回傳 PyTorch 張量
    )
    print("input_ids 形狀      :", tuple(out["input_ids"].shape), " (B, L)")
    print("attention_mask 形狀 :", tuple(out["attention_mask"].shape))
    print("\nattention_mask（1=真實 token, 0=padding）：")
    print(out["attention_mask"])

**解讀**：三句被打包成 `(3, L)` 的整數張量，短句尾端補 0（padding），
`attention_mask` 讓模型「只看真實 token」。這個 `(B, L)` 張量就是餵進 Transformer 的標準輸入。

## 5. 從零訓練一個 BPE Tokenizer（理解詞彙表怎麼長出來）

用 `tokenizers` 套件在自備小語料上訓練 BPE，看詞彙表如何由合併規則建立。**無需下載任何模型**。

In [ ]:
if HAS_HF:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    mini_corpus = [
        "low lower lowest",
        "new newer newest",
        "wide wider widest",
        "tokenization tokenizer tokenize",
    ]
    bpe = Tokenizer(BPE(unk_token="[UNK]"))
    bpe.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=60, special_tokens=["[UNK]", "[PAD]"])
    bpe.train_from_iterator(mini_corpus, trainer=trainer)

    print("學到的詞彙表大小：", bpe.get_vocab_size())
    sample = bpe.encode("the lowest tokenizer")
    print("切分結果：", sample.tokens)
    print("（注意：訓練語料只看過 low/lower/lowest…，'the' 這種生字會被拆成已知碎片）")

**解讀**：BPE 從字元開始，把「最常一起出現」的碎片不斷合併成 subword。
因為任何字串都能退回字元層級拼出來，所以面對訓練時沒見過的字也**不會 OOV**。

## 6. Token 預算：token ≠ 單字（成本直覺）

大模型的「上下文長度」「API 計費」都以 **token** 計。一個經驗法則（英文）：
**1 token ≈ 0.75 個英文單字 ≈ 4 個字元**；中文常常 **1 字 ≈ 1～2 token**。

In [ ]:
if HAS_HF:
    samples = [
        "Hello world!",
        "資料前處理是大模型訓練的第一步。",
        "Antidisestablishmentarianism",
    ]
    for s in samples:
        n = len(tok(s)["input_ids"])
        print(f"{n:>3} tokens | {len(s):>3} 字元 | {s}")
    print("\n→ 中文、長生字會用較多 token；估算資料量/成本時要用 token 數，不是字數。")

## 小結

| 重點 | 內容 |
|:--|:--|
| 標準輸出 | `input_ids (B, L)` + `attention_mask (B, L)`（整數張量） |
| 為何 subword | 解決 OOV、控制詞彙表、跨語言，現代模型一律採用 |
| 三大演算法 | BPE（GPT/Llama）、WordPiece（BERT）、Unigram/SentencePiece（T5/多語） |
| 實務 | 用 `AutoTokenizer` 自動對應模型；`padding`/`truncation` 產生 `(B, L)` |
| 成本 | 以 token 計，token ≠ 單字 |

**下一節 `03_contextual_embeddings`**：把 `input_ids` 餵進 Transformer，
得到「**隨上下文變化**」的嵌入向量——徹底解決經典靜態詞向量「一詞一向量」的限制。